# Analyse des données d'occulométrie
Ce notebook orchestre le pipeline complet d'analyse. Il extrait d'abord les statistiques descriptives des fixations brutes, puis lance les pipelines d'évaluation des similarités et des prédictivités.

## 1. Paths

### Root folder

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(ROOT)

/home/larbez/Projects/DeepGaze


### Access paths

In [2]:
# Access Paths
data_dir = ROOT / "DATA_OCCULO_2026" / "2026" / "paired" 
images_dir = data_dir / "images"
output_dir = ROOT / "outputs" 

metadata_json = data_dir / "metadata" / "metadata.json"
annotations_json = data_dir / "metadata" / "annotations.json"
visible_path = data_dir / "dataframes" / "fixations_visible.csv" 
infrared_path = data_dir / "dataframes" / "fixations_infrared.csv" 
infrared_centerbias_path = data_dir / "centerbias" / "infrared.npy"
visible_centerbias_path = data_dir / "centerbias" / "visible.npy"

### Importations

In [3]:
from analysis.saver import Saver
from analysis.comparator import Comparator
from analysis.analyzer import Analyzer
from helpers.loader import Loader

## 2. Analysis

In [4]:
loader = Loader(metadata_json)
saver = Saver(output_dir)

visible_df = loader.load_dataframe(visible_path)
infrared_df = loader.load_dataframe(infrared_path)

### Step 1 : Comparison

In [5]:
comparator = Comparator(
    metadata_json=metadata_json,
    annotations_json=annotations_json,
    sigma_deg=1.0,
    viewing_distance_cm=70.0,
    screen_width_cm=53.1,
    screen_width_px=1360,
)

#### Modality Comparisons

In [7]:
expe1_results = comparator.run_expe1_old(
    infrared_df=infrared_df,
    visible_df=visible_df,
    n_permutations=5,
    seed=42,
)

Running human-human analysis: 100%|██████████| 150/150 [02:47<00:00,  1.11s/it]


In [8]:
output_dir_expe1 = output_dir / "expe1" / "comparison"
expe1_df = saver.save_expe1_results(
    results=expe1_results, 
    output_dir=output_dir_expe1
)

#### DeepGaze Comparisons

In [ ]:
expe2_results = comparator.run_expe2(
    infrared_df=infrared_df,
    visible_df=visible_df,
    infrared_image_dir=images_dir / "infrared",
    visible_image_dir=images_dir / "visible",
    infrared_centerbias_path=infrared_centerbias_path,
    visible_centerbias_path=visible_centerbias_path,
)

/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded pretrained weights for efficientnet-b5


Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the 

Loaded pretrained weights for efficientnet-b5


Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
Comparing DeepGaze RGB and IR: 100%|██████████| 150/150 [05:34<00:00,  2.23s/it]


In [ ]:
output_dir_expe2 = output_dir / "expe2" / "comparison"
expe2_df = saver.save_expe2_results(
    results=expe2_results, 
    output_dir=output_dir_expe2
)

### Step 2: Statistical Analysis

In [ ]:
analyzer = Analyzer(alpha=0.05, correction_method="holm")

#### Modality Analysis

In [ ]:
expe1_df = loader.load_dataframe(ROOT / 'outputs' / 'expe1' / 'comparison' / 'all_results.csv')
expe1_analysis = analyzer.analyze_expe1(expe1_df) 
saver.save_expe1_analysis(expe1_analysis)

,metric_key,metric_type,metric_name,test_name,test_type,alternative,n,mean_x,mean_y,mean_diff,...,p_value,cohen_d,CI95,power,mean_score_inter,mean_score_perm,mean_delta_perm_inter,p_value_corrected,significant_corrected,correction_method
0,similarity_cc,similarity,cc,perm_vs_inter,one-sample,greater,150,0.114747,0.0,0.114747,...,1.178454e-33,1.280602,"[0.1, null]",1.0,0.825392,0.940139,0.114747,1.178454e-33,True,holm
1,similarity_sim,similarity,sim,perm_vs_inter,one-sample,greater,150,0.108922,0.0,0.108922,...,3.605559e-53,1.952485,"[0.1, null]",1.0,0.710234,0.819156,0.108922,2.163335e-52,True,holm
2,similarity_kl,similarity,kl,perm_vs_inter,one-sample,greater,150,0.296296,0.0,0.296296,...,4.092660e-38,1.423933,"[0.27, null]",1.0,-0.523619,-0.227323,0.296296,2.046330e-37,True,holm
3,predictivity_nss,predictivity,nss,perm_vs_inter,one-sample,greater,150,0.263731,0.0,0.263731,...,8.623969e-38,1.413368,"[0.24, null]",1.0,2.353846,2.617577,0.263731,3.449587e-37,True,holm
4,predictivity_auc_judd,predictivity,auc_judd,perm_vs_inter,one-sample,greater,150,0.022550,0.0,0.022550,...,4.376061e-34,1.294238,"[0.02, null]",1.0,0.881039,0.903589,0.022550,8.752121e-34,True,holm
5,predictivity_ig,predictivity,ig,perm_vs_inter,one-sample,greater,150,0.264464,0.0,0.264464,...,1.032546e-36,1.378369,"[0.24, null]",1.0,1.111922,1.376386,0.264464,3.097637e-36,True,holm


#### Deepgaze Analysis

In [ ]:
expe2_df = loader.load_dataframe(ROOT / 'outputs' / 'expe2' / 'comparison' / 'all_results.csv')
expe2_analysis = analyzer.analyze_expe2(expe2_df) 
saver.save_expe2_analysis(expe2_analysis)

,metric_key,metric_type,metric_name,test_name,test_type,alternative,n,mean_x,mean_y,mean_diff,...,cohen_d,CI95,power,BF10,mean_score_rgb,mean_score_ir,mean_delta_rgb_ir,p_value_corrected,significant_corrected,correction_method
0,similarity_cc,similarity,cc,rgb_vs_ir,one-sample,two-sided,150,-0.015955,0.0,-0.015955,...,0.155966,"[-0.03, 0.0]",0.475302,0.534,0.784524,0.800480,-0.015955,0.116060,False,holm
1,similarity_sim,similarity,sim,rgb_vs_ir,one-sample,two-sided,150,0.014193,0.0,0.014193,...,0.235538,"[0.0, 0.02]",0.817567,4.848,0.673560,0.659366,0.014193,0.022494,True,holm
2,similarity_kl,similarity,kl,rgb_vs_ir,one-sample,two-sided,150,0.022868,0.0,0.022868,...,0.181626,"[0.0, 0.04]",0.598775,0.992,-0.373551,-0.396419,0.022868,0.082864,False,holm
3,predictivity_nss,predictivity,nss,rgb_vs_ir,one-sample,two-sided,150,-0.144563,0.0,-0.144563,...,0.334183,"[-0.21, -0.07]",0.982415,221.602,2.015453,2.160017,-0.144563,0.000417,True,holm
4,predictivity_auc_judd,predictivity,auc_judd,rgb_vs_ir,one-sample,two-sided,150,-0.006964,0.0,-0.006964,...,0.214673,"[-0.01, -0.0]",0.742884,2.510,0.880758,0.887721,-0.006964,0.037825,True,holm
5,predictivity_ig,predictivity,ig,rgb_vs_ir,one-sample,two-sided,150,-0.016622,0.0,-0.016622,...,0.079712,"[-0.05, 0.02]",0.162789,0.145,1.101654,1.118276,-0.016622,0.330511,False,holm
